## 06 - Supplementary Figures S1 and S2

Creates the descriptive market-size and distribution figures used in the supplement. Listings are deduplicated by `Website` and `Pages`; reviews are counted as raw review rows. Vendor breadth uses Category 3 product breadth and `Category 2 (most common)` breadth so it matches the product-space notebooks.


In [ ]:
# --- config ---
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
import dnm_viz_utils as dvu

MARKETS = dvu.MARKETS
PANEL_BY_MARKET = dvu.PANEL_BY_MARKET
load_master_drugs = dvu.load_master_drugs
load_category2_most_common_mapping = dvu.load_category2_most_common_mapping
repo_root = dvu.repo_root
style_axes = dvu.style_axes

os.chdir(repo_root())
ROOT = repo_root()
FIG = ROOT
TAB = ROOT / "prescription_reports/new_category2_networks/tables"
TAB.mkdir(parents=True, exist_ok=True)

SITE_COLORS = {
    "WhiteHouse Market": "#D43E88",
    "Dark0de": "#0D2931",
    "ToRReZ": "#BEA39F",
    "Versus": "#F6C8B0",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
})
print("Repo:", ROOT)
print("Markets:", MARKETS)


In [ ]:
def add_listing_id(df: pd.DataFrame) -> pd.Series:
    website = df["Website"].fillna("").astype(str).str.strip()
    pages = df["Pages"].fillna("").astype(str).str.strip()
    vendor = df["ID_Num"].fillna("").astype(str).str.strip()
    names = df["Productname"].fillna("").astype(str).str.strip().str.casefold()
    cat3 = df["Category 3"].fillna("").astype(str).str.strip()
    fallback = website + " | " + vendor + " | " + names + " | " + cat3
    valid_pages = pages.ne("") & ~pages.str.lower().isin(["nan", "none"])
    out = pages.where(valid_pages, fallback)
    return out.where(~website.isin(["WhiteHouse Market", "ToRReZ"]), fallback).astype(str).str.strip()


def prepare_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    df = load_master_drugs()
    df = df[df["Website"].isin(MARKETS)].copy()
    df["listing_id"] = add_listing_id(df)
    df["Category 3"] = df["Category 3"].astype(str).str.strip()
    prod_to_cat2 = load_category2_most_common_mapping(df)
    df["cat2_most_common"] = df["Category 3"].map(prod_to_cat2).fillna("Unknown")

    listing_df = df.drop_duplicates(["Website", "listing_id"]).copy()
    for col in ["Amount (grams)", "ProductPrice"]:
        if col in listing_df.columns:
            listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")
    listing_df["price_per_gram"] = listing_df["ProductPrice"] / listing_df["Amount (grams)"]
    listing_df.loc[~np.isfinite(listing_df["price_per_gram"]), "price_per_gram"] = np.nan
    listing_df.loc[listing_df["price_per_gram"] <= 0, "price_per_gram"] = np.nan
    listing_df.loc[listing_df["Amount (grams)"] <= 0, "Amount (grams)"] = np.nan
    return df, listing_df


df_reviews, df_listings = prepare_data()
print(f"Review rows: {len(df_reviews):,}")
print(f"Distinct listings: {len(df_listings):,}")
df_listings.head(2)


In [ ]:
def market_summary(df_reviews: pd.DataFrame, df_listings: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for market in MARKETS:
        rev = df_reviews[df_reviews["Website"] == market]
        lis = df_listings[df_listings["Website"] == market]
        rows.append({
            "Website": market,
            "n_listings": int(lis["listing_id"].nunique()),
            "n_vendors": int(lis["ID_Num"].nunique()),
            "n_reviews": int(len(rev)),
            "median_listing_size_g": float(lis["Amount (grams)"].median()),
            "median_unit_price": float(lis["price_per_gram"].median()),
        })
    return pd.DataFrame(rows)


def vendor_breadth(df_listings: pd.DataFrame) -> pd.DataFrame:
    d = df_listings.dropna(subset=["Website", "ID_Num"]).copy()
    out = (
        d.groupby(["Website", "ID_Num"])
        .agg(
            breadth_cat3=("Category 3", "nunique"),
            breadth_cat2=("cat2_most_common", "nunique"),
            n_listings=("listing_id", "nunique"),
        )
        .reset_index()
    )
    return out


summary = market_summary(df_reviews, df_listings)
breadth = vendor_breadth(df_listings)
summary.to_csv(TAB / "website_market_summary.csv", index=False)
breadth.to_csv(TAB / "website_vendor_breadth.csv", index=False)
print(summary)
print(breadth[["breadth_cat2", "breadth_cat3", "n_listings"]].describe())


In [ ]:
def kfmt(x, _pos=None):
    if abs(x) >= 1_000_000:
        return f"{x/1_000_000:g}M"
    if abs(x) >= 1_000:
        return f"{x/1_000:g}k"
    return f"{x:g}"


def plot_market_size(summary: pd.DataFrame):
    fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.1), constrained_layout=True, sharey=True)
    metrics = [
        ("n_listings", "Reviewed listing proxies", "A"),
        ("n_vendors", "Unique vendors", "B"),
        ("n_reviews", "Reviews", "C"),
    ]
    order = summary.sort_values("n_reviews", ascending=False)["Website"].tolist()
    smap = summary.set_index("Website").loc[order].reset_index()
    y = np.arange(len(smap))
    for ax, (col, xlabel, letter) in zip(axes, metrics):
        ax.barh(y, smap[col], color=[SITE_COLORS[m] for m in smap["Website"]], edgecolor="0.25", linewidth=0.7)
        ax.set_yticks(y)
        ax.set_yticklabels(smap["Website"])
        ax.invert_yaxis()
        ax.set_xlabel(xlabel)
        ax.text(-0.18, 1.02, letter, transform=ax.transAxes, ha="left", va="bottom", fontsize=16)
        ax.xaxis.set_major_formatter(FuncFormatter(kfmt))
        style_axes(ax)
    for ax in axes[1:]:
        ax.tick_params(axis="y", left=False, labelleft=False)
    return fig


fig = plot_market_size(summary)
fig.savefig(FIG / "marketsize.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved", FIG / "marketsize.png")


In [ ]:
def ecdf(ax, values, *, label=None, color=None, logx=False):
    x = pd.Series(values).dropna().astype(float)
    x = x[np.isfinite(x)]
    if logx:
        x = x[x > 0]
    if x.empty:
        return
    xs = np.sort(x.to_numpy())
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax.plot(xs, ys, lw=1.7, label=label, color=color)


def binned_median(x, y, bins=12):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    d = d[(d["x"] > 0) & (d["y"] > 0)]
    if len(d) < 20:
        return np.array([]), np.array([])
    edges = np.logspace(np.log10(d["x"].min()), np.log10(d["x"].max()), bins)
    d["bin"] = pd.cut(d["x"], edges, include_lowest=True)
    out = d.groupby("bin", observed=True).agg(x=("x", "median"), y=("y", "median"), n=("y", "size")).reset_index(drop=True)
    out = out[out["n"] >= 5]
    return out["x"].to_numpy(), out["y"].to_numpy()


def plot_market_distributions(df_listings: pd.DataFrame, breadth: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(9, 6.2), constrained_layout=True)
    axes = axes.flatten()

    for market in MARKETS:
        color = SITE_COLORS[market]
        lis = df_listings[df_listings["Website"] == market]
        br = breadth[breadth["Website"] == market]
        ecdf(axes[0], lis["Amount (grams)"], label=market, color=color, logx=True)
        ecdf(axes[1], lis["price_per_gram"], label=market, color=color, logx=True)
        ecdf(axes[3], br["breadth_cat2"], label=market, color=color)
        bx, by = binned_median(lis["Amount (grams)"], lis["price_per_gram"])
        if len(bx):
            axes[2].plot(bx, by, lw=1.7, color=color, label=market)

    labels = [
        ("A", "Listing size (g)", "ECDF"),
        ("B", "Unit price", "ECDF"),
        ("C", "Listing size (g)", "Median unit price"),
        ("D", "Distinct Category 2 per vendor", "ECDF"),
    ]
    for ax, (letter, xlabel, ylabel) in zip(axes, labels):
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.text(-0.12, 1.02, letter, transform=ax.transAxes, ha="left", va="bottom", fontsize=16)
        style_axes(ax)
    for ax in axes[:3]:
        ax.set_xscale("log")
    axes[1].set_xscale("log")
    axes[2].set_yscale("log")
    axes[1].legend(frameon=False, fontsize=8)
    return fig


fig = plot_market_distributions(df_listings, breadth)
fig.savefig(FIG / "marketdistributions.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved", FIG / "marketdistributions.png")
